In [1]:
# input
metal_sites = "../nr_metal_sites.tsv"
pdb_files_lookup = "../tmp/mbp_files.tsv"
# output
coord_atom_dist = "./tmp/coord_atom_dist.tsv"

In [2]:
import pandas as pd
import os
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB.Residue import Residue


def parse_mmcif_file(
    id: str,
    file: str,
    target_residue_ids: set
) -> dict[tuple, Residue]:

    file = os.path.expanduser(file)
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure(id, file)
    residues = structure.get_residues()

    id2residue = dict()
    for r in residues:
        r: Residue
        full_id = r.get_full_id()
        id = (full_id[2], full_id[3][1], full_id[3][2])
        if full_id[3][0] == " " and id in target_residue_ids:
            id2residue[id] = r
    return id2residue


allowed_coord_atom = {
    ("Cys", "SG"),
    ("Asp", "OD1"),
    ("Asp", "OD2"),
    ("Glu", "OE1"),
    ("Glu", "OE2"),
    ("His", "ND1"),
    ("His", "NE2"),
}

def get_resi_id(row): 
    return (row['resi_chain'], row['resi_pdb_seq_num'], row['resi_pdb_ins_code'])


def filter_df(
    df: pd.DataFrame,
):
    df_s = df.copy()
    df_s = df_s[df_s.apply(lambda row: (row['resi'], row['atom']) in allowed_coord_atom, axis=1)]
    ids = set(df_s.apply(lambda row: get_resi_id(row), axis=1))
    if len(ids) < 3:
        return None
    return df_s

In [3]:

from tqdm import tqdm
from itertools import combinations


records = []

df = pd.read_table(pdb_files_lookup)
pdb2file = dict(zip(df['pdb'], df['pdb_file']))
df = pd.read_table(metal_sites)
for pdb, df_pdb in tqdm(df.groupby(by=['pdb'])):
    f = pdb2file[pdb]
    residue_ids = set(df_pdb.apply(lambda row: get_resi_id(row), axis=1))
    id2residue = parse_mmcif_file(pdb, pdb2file[pdb], residue_ids)
    
    for site_info, df_site in df_pdb.groupby(by=['metal_chain', 'metal_pdb_seq_num', 'metal_resi']):
        df_s = filter_df(df_site)
        if df_s is None: continue

        for row_i, row_j in combinations(df_s.to_dict('records'), 2):
            atom_i = id2residue[get_resi_id(row_i)][row_i['atom']]
            atom_j = id2residue[get_resi_id(row_j)][row_j['atom']]
            dist = atom_i - atom_j

            records.append({
                "pdb": pdb,
                "metal_chain": site_info[0],
                "metal_pdb_seq_num": site_info[1],
                "metal_resi": site_info[2],
                "resi_i": row_i['resi'],
                "resi_chain_i": row_i['resi_chain'],
                "resi_pdb_seq_num_i": row_i['resi_pdb_seq_num'],
                "resi_pdb_ins_code_i": row_i['resi_pdb_ins_code'],
                "atom_i": row_i['atom'],
                "resi_j": row_j['resi'],
                "resi_chain_j": row_j['resi_chain'],
                "resi_pdb_seq_num_j": row_j['resi_pdb_seq_num'],
                "resi_pdb_ins_code_j": row_j['resi_pdb_ins_code'],
                "atom_j": row_j['atom'],
                "dist": dist
            })
# half an hour

  0%|          | 0/4612 [00:00<?, ?it/s]/home/zhangf/install/miniforge3/envs/metalnet2/lib/python3.9/site-packages/tqdm/std.py:1178: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  for obj in iterable:
100%|██████████| 4612/4612 [36:44<00:00,  2.09it/s]  


In [ ]:
pd.DataFrame(records).to_csv(coord_atom_dist, sep="\t", index=None)

### analysis

In [37]:
df = pd.read_table(coord_atom_dist)
df_ion = df[df['metal_resi'].map(lambda x: x not in {"FES", "SF4", "F3S"})]
df_fes = df[df['metal_resi'].map(lambda x: x in {"FES", "SF4", "F3S"})]
df_ion['dist'].quantile(0.95)
df_fes['dist'].quantile(0.95)
df_ion['dist'].quantile(0.05)
df_fes['dist'].quantile(0.05)

4.6623917200000005

6.999775825

2.2064337000000003

3.6711183099999998

In [30]:
df_ion.describe()
df_fes.describe()

,metal_pdb_seq_num,resi_pdb_seq_num_i,resi_pdb_seq_num_j,dist
count,28545.000000,28545.000000,28545.000000,28545.000000
mean,708.180592,314.321527,313.758942,3.548735
std,877.614647,482.656371,480.656770,0.624854
min,1.000000,-20.000000,-22.000000,0.000000
25%,301.000000,87.000000,88.000000,3.171997
50%,501.000000,184.000000,183.000000,3.549144
75%,806.000000,359.000000,359.000000,3.890219
max,9203.000000,7101.000000,7101.000000,6.800351


,metal_pdb_seq_num,resi_pdb_seq_num_i,resi_pdb_seq_num_j,dist
count,2288.000000,2288.000000,2288.000000,2288.000000
mean,702.086538,185.697552,185.031031,6.116556
std,837.447670,181.628280,179.921059,0.977801
min,61.000000,3.000000,3.000000,0.000000
25%,301.000000,57.000000,59.000000,6.018828
50%,501.000000,126.000000,126.000000,6.418303
75%,801.000000,244.000000,244.000000,6.643054
max,5804.000000,1148.000000,1143.000000,8.020123


In [38]:
# two sphere-metal ion CUA
df_ion[df_ion['dist'].map(lambda x: x > 6)]

# 1ile: coord cys modeled as ss-bond
df_ion[df_ion['dist'].map(lambda x: x < 1)]

,pdb,metal_chain,metal_pdb_seq_num,metal_resi,resi_i,resi_chain_i,resi_pdb_seq_num_i,resi_pdb_ins_code_i,atom_i,resi_j,resi_chain_j,resi_pdb_seq_num_j,resi_pdb_ins_code_j,atom_j,dist
1539,1qle,B,301,CUA,His,B,181,,ND1,His,B,224,,ND1,6.374971
3367,2cua,B,170,CUA,His,B,114,,ND1,His,B,157,,ND1,6.214504
5877,2yev,E,585,CUA,His,E,205,,ND1,His,E,162,,ND1,6.323862
23312,6ptt,B,201,CUA,His,B,157,,ND1,His,B,114,,ND1,6.800351


,pdb,metal_chain,metal_pdb_seq_num,metal_resi,resi_i,resi_chain_i,resi_pdb_seq_num_i,resi_pdb_ins_code_i,atom_i,resi_j,resi_chain_j,resi_pdb_seq_num_j,resi_pdb_ins_code_j,atom_j,dist
224,1cyx,A,316,CUA,Cys,A,207,,SG,Cys,A,207,,SG,0.000000
225,1cyx,A,316,CUA,Cys,A,211,,SG,Cys,A,211,,SG,0.000000
716,1ile,A,1102,ZN,Cys,A,461,,SG,Cys,A,464,,SG,0.942861
719,1ile,A,1102,ZN,Cys,A,502,,SG,Cys,A,504,,SG,0.926747
1531,1qle,B,301,CUA,Cys,B,220,,SG,Cys,B,220,,SG,0.000000
1532,1qle,B,301,CUA,Cys,B,216,,SG,Cys,B,216,,SG,0.000000
3374,2cua,B,170,CUA,Cys,B,149,,SG,Cys,B,149,,SG,0.000000
3378,2cua,B,170,CUA,Cys,B,153,,SG,Cys,B,153,,SG,0.000000
5886,2yev,E,585,CUA,Cys,E,201,,SG,Cys,E,201,,SG,0.000000
5891,2yev,E,585,CUA,Cys,E,197,,SG,Cys,E,197,,SG,0.000000


In [39]:
df_fes[df_fes['dist'].map(lambda x: x < 1)]

,pdb,metal_chain,metal_pdb_seq_num,metal_resi,resi_i,resi_chain_i,resi_pdb_seq_num_i,resi_pdb_ins_code_i,atom_i,resi_j,resi_chain_j,resi_pdb_seq_num_j,resi_pdb_ins_code_j,atom_j,dist
29523,7z6q,A,821,SF4,Cys,A,536,,SG,Cys,A,536,,SG,0.0
29528,7z6q,A,821,SF4,Cys,a,536,,SG,Cys,a,536,,SG,0.0
30084,8b6j,e,301,FES,Cys,e,218,,SG,Cys,e,218,,SG,0.0
